In [7]:
%pip install openpyxl


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import re
import pandas as pd

# 1. Load the Datasets
# Assuming 'movies.csv' and 'ratings.xlsx' are in your current directory
movies_df = pd.read_csv("movies.csv")
ratings_df = pd.read_excel("ratings.xlsx")


In [9]:

# 2. Merge the Data
# Merging on the shared 'movieId' column
merged_df = pd.merge(movies_df, ratings_df, on="movieId")


In [10]:

# 3. Clean the Data
# Remove rows with missing values
merged_df = merged_df.dropna()

# Convert data types: Convert timestamp to readable datetime format
# (Handles cases where timestamp might be in Unix epochs)
if "timestamp" in merged_df.columns:
    merged_df["timestamp"] = pd.to_datetime(merged_df["timestamp"], unit="s")

# Extract the release year from the title column (e.g., "Toy Story (1995)" -> 1995)
# This regex looks for 4 digits enclosed in parentheses at the end of the string
merged_df["release_year"] = merged_df["title"].str.extract(r"\((\d{4})\)$")
merged_df["release_year"] = pd.to_numeric(
    merged_df["release_year"], errors="coerce"
)


In [11]:


# 4. Analysis
# --- Question 1: Which movie has the highest average rating? (Min 100 ratings) ---

# Group by title and calculate both the count and mean of ratings
movie_stats = (
    merged_df.groupby("title")["rating"].agg(["count", "mean"]).reset_index()
)

# Filter for movies with at least 100 ratings
popular_movies = movie_stats[movie_stats["count"] >= 100]

# Find the movie with the highest average rating
highest_rated_movie = popular_movies.sort_values(
    by="mean", ascending=False
).iloc[0]

print("--- Question 1 Result ---")
print(
    f"Movie: {highest_rated_movie['title']}\n"
    f"Average Rating: {highest_rated_movie['mean']:.2f}\n"
    f"Total Ratings: {int(highest_rated_movie['count'])}\n"
)



--- Question 1 Result ---
Movie: Shawshank Redemption, The (1994)
Average Rating: 4.43
Total Ratings: 317



In [12]:

# --- Question 2: Which genre of movies tends to get the highest ratings? ---

# First, split the genres (separated by '|') and explode them into individual rows
# This duplicates the rating for each genre a movie belongs to
exploded_genres_df = merged_df.copy()
exploded_genres_df["genres"] = exploded_genres_df["genres"].str.split("|")
exploded_genres_df = exploded_genres_df.explode("genres")

# Group by the individual genres and calculate the average rating
genre_stats = (
    exploded_genres_df.groupby("genres")["rating"]
    .agg(["count", "mean"])
    .reset_index()
)

# Sort to find the highest rated genre
highest_rated_genre = genre_stats.sort_values(by="mean", ascending=False).iloc[
    0
]

print("--- Question 2 Result ---")
print(
    f"Genre: {highest_rated_genre['genres']}\n"
    f"Average Rating: {highest_rated_genre['mean']:.2f}\n"
)

--- Question 2 Result ---
Genre: Film-Noir
Average Rating: 3.92

